<a href="https://colab.research.google.com/github/gerardbullock/gerardbullock.github.io/blob/main/Movie_Recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [80]:
#import libraries
import numpy as np
import pandas as pd
import sklearn

In [81]:
#import data
movies = pd.read_csv('/content/movies.csv')
movies.head()


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [82]:
#remove spaces | dashes | parenthesis | characters or digits from title column
import re

def clean_title(title):
    title = re.sub("[^a-zA-Z0-9 ]", "", title)
    return title

In [83]:
movies["clean_title"]= movies["title"].apply(clean_title)
movies.head()

,movieId,title,genres,clean_title
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story 1995
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji 1995
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men 1995
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale 1995
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II 1995


In [84]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(ngram_range=(1,2))

tfidf = vectorizer.fit_transform(movies["clean_title"])
tfidf

<62423x170073 sparse matrix of type '<class 'numpy.float64'>'
	with 446566 stored elements in Compressed Sparse Row format>

In [85]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

#def search(title):
title = "Harry Potter"
title = clean_title(title)
query_vec = vectorizer.transform([title])
query_vec

<1x170073 sparse matrix of type '<class 'numpy.float64'>'
	with 3 stored elements in Compressed Sparse Row format>

In [86]:
#find similarity between search terms and titles

#def search(title):
title = "Harry Potter"
title = clean_title(title)
query_vec = vectorizer.transform([title])
similarity = cosine_similarity(query_vec, tfidf).flatten()
similarity

array([0., 0., 0., ..., 0., 0., 0.])

In [87]:
#no matches to title Harry Potter 

In [88]:
title = "Men 1995"
title = clean_title(title)
query_vec = vectorizer.transform([title])
similarity = cosine_similarity(query_vec, tfidf).flatten()
indicies = np.argpartition(similarity, -5)[-5:]
indicies

array([ 7071, 11003, 28489,     2, 60174])

In [89]:
#return top 5 most similar movies
title = "Men 1995"
title = clean_title(title)
query_vec = vectorizer.transform([title])
similarity = cosine_similarity(query_vec, tfidf).flatten()
indicies = np.argpartition(similarity, -5)[-5:]
results = movies.iloc[indicies]
results

,movieId,title,genres,clean_title
7071,7196,"Men, The (1950)",Drama,Men The 1950
11003,47484,G Men (1935),Crime|Drama,G Men 1935
28489,131824,Men... (1985),Comedy,Men 1985
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men 1995
60174,202701,Любить по-русски (1995),Drama|Romance,1995


In [90]:
#top match of most similar movies
title = "Men 1995"
title = clean_title(title)
query_vec = vectorizer.transform([title])
similarity = cosine_similarity(query_vec, tfidf).flatten()
indicies = np.argpartition(similarity, -5)[-5:]
results = movies.iloc[indicies][::-1]
results

,movieId,title,genres,clean_title
60174,202701,Любить по-русски (1995),Drama|Romance,1995
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men 1995
28489,131824,Men... (1985),Comedy,Men 1985
11003,47484,G Men (1935),Crime|Drama,G Men 1935
7071,7196,"Men, The (1950)",Drama,Men The 1950


In [91]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def search(title):
    title = clean_title(title)
    query_vec = vectorizer.transform([title])
    similarity = cosine_similarity(query_vec, tfidf).flatten()
    indices = np.argpartition(similarity, -5)[-5:]
    results = movies.iloc[indices].iloc[::-1]

In [92]:
results

,movieId,title,genres,clean_title
60174,202701,Любить по-русски (1995),Drama|Romance,1995
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men 1995
28489,131824,Men... (1985),Comedy,Men 1985
11003,47484,G Men (1935),Crime|Drama,G Men 1935
7071,7196,"Men, The (1950)",Drama,Men The 1950


In [93]:
import ipywidgets as widgets
from IPython.display import display

movie_input = widgets.Text(
      value = "Toy Story",
      description = "Movie Title:",
      disabled=False
)
movie_input

Text(value='Toy Story', description='Movie Title:')

In [94]:
import ipywidgets as widgets
from IPython.display import display

movie_input = widgets.Text(
      value = "Toy Story",
      description = "Movie Title:",
      disabled=False
)
movie_list = widgets.Output()

def on_type(data):
    with movie_list:
      movie_list.clear_output()
      title = data["new"]
      if len(title) > 5:
        display(search(title))

movie_input

Text(value='Toy Story', description='Movie Title:')

In [95]:
import ipywidgets as widgets
from IPython.display import display

movie_input = widgets.Text(
    value='Toy Story',
    description='Movie Title:',
    disabled=False
)
movie_list = widgets.Output()

def on_type(data):
    with movie_list:
        movie_list.clear_output()
        title = data["new"]
        if len(title) > 5:
            display(search(title))

movie_input.observe(on_type, names='value')


display(movie_input, movie_list)

Text(value='Toy Story', description='Movie Title:')

Output()

In [96]:
import ipywidgets as widgets
from IPython.display import display

movie_input = widgets.Text(
      value = (title),
      description = "Movie Title:",
      disabled=False
)
movie_list = widgets.Output()

def on_type(data):
    with movie_list:
      movie_list.clear_output()
      title = data["new"]
      if len(title) > 5:
        display(search(title))

movie_input.observe(on_type, names='values')
display(movie_input, movie_list)

Text(value='Men 1995', description='Movie Title:')

Output()

In [97]:
movie_id = 89745

#def find_similar_movies(movie_id):
movie = movies[movies["movieId"] == movie_id]

In [98]:
ratings = pd.read_csv("/content/ratings.csv")

In [99]:
ratings.dtypes

userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object

In [100]:
similar_users = ratings[(ratings["movieId"] == movie_id) & (ratings["rating"] > 4)]["userId"].unique()

In [101]:
similar_user_recs = ratings[(ratings["userId"].isin(similar_users)) & (ratings["rating"] > 4)]["movieId"]

In [102]:
similar_user_recs = similar_user_recs.value_counts() / len(similar_users)

similar_user_recs = similar_user_recs[similar_user_recs > .10]

In [103]:
all_users = ratings[(ratings["movieId"].isin(similar_user_recs.index)) & (ratings["rating"] > 4)]

In [104]:
all_user_recs = all_users["movieId"].value_counts() / len(all_users["userId"].unique())

In [105]:
rec_percentages = pd.concat([similar_user_recs, all_user_recs], axis=1)
rec_percentages.columns = ["similar", "all"]

In [106]:
rec_percentages

,similar,all
1,0.236083,0.126250
32,0.103877,0.101516
47,0.203115,0.146232
50,0.211067,0.202959
110,0.182240,0.162835
...,...,...
134853,0.198641,0.036444
152081,0.133532,0.020652
164179,0.128728,0.029124
166528,0.124751,0.014411


In [107]:
rec_percentages["score"] = rec_percentages["similar"] / rec_percentages["all"]

In [108]:
rec_percentages = rec_percentages.sort_values("score", ascending=False)

In [109]:
rec_percentages.head(10).merge(movies, left_index=True, right_on="movieId")

,similar,all,score,movieId,title,genres,clean_title
17067,1.000000,0.040459,24.716368,89745,"Avengers, The (2012)",Action|Adventure|Sci-Fi|IMAX,Avengers The 2012
20513,0.103711,0.005289,19.610199,106072,Thor: The Dark World (2013),Action|Adventure|Fantasy|IMAX,Thor The Dark World 2013
25058,0.241054,0.012367,19.491770,122892,Avengers: Age of Ultron (2015),Action|Adventure|Sci-Fi,Avengers Age of Ultron 2015
19678,0.216534,0.012119,17.867419,102125,Iron Man 3 (2013),Action|Sci-Fi|Thriller|IMAX,Iron Man 3 2013
16725,0.215043,0.012052,17.843074,88140,Captain America: The First Avenger (2011),Action|Adventure|Sci-Fi|Thriller|War,Captain America The First Avenger 2011
16312,0.175447,0.010142,17.299824,86332,Thor (2011),Action|Adventure|Drama|Fantasy|IMAX,Thor 2011
21348,0.287608,0.016737,17.183667,110102,Captain America: The Winter Soldier (2014),Action|Adventure|Sci-Fi|IMAX,Captain America The Winter Soldier 2014
25071,0.214049,0.012856,16.649399,122920,Captain America: Civil War (2016),Action|Sci-Fi|Thriller,Captain America Civil War 2016
25061,0.136017,0.008573,15.865628,122900,Ant-Man (2015),Action|Adventure|Sci-Fi,AntMan 2015
14628,0.242876,0.015517,15.651921,77561,Iron Man 2 (2010),Action|Adventure|Sci-Fi|Thriller|IMAX,Iron Man 2 2010


In [110]:
def find_similar_movies(movie_id):
    similar_users = ratings[(ratings["movieId"] == movie_id) & (ratings["rating"] > 4)]["userId"].unique()
    similar_user_recs = ratings[(ratings["userId"].isin(similar_users)) & (ratings["rating"] > 4)]["movieId"]
    similar_user_recs = similar_user_recs.value_counts() / len(similar_users)

    similar_user_recs = similar_user_recs[similar_user_recs > .10]
    all_users = ratings[(ratings["movieId"].isin(similar_user_recs.index)) & (ratings["rating"] > 4)]
    all_user_recs = all_users["movieId"].value_counts() / len(all_users["userId"].unique())
    rec_percentages = pd.concat([similar_user_recs, all_user_recs], axis=1)
    rec_percentages.columns = ["similar", "all"]
    
    rec_percentages["score"] = rec_percentages["similar"] / rec_percentages["all"]
    rec_percentages = rec_percentages.sort_values("score", ascending=False)
    return rec_percentages.head(10).merge(movies, left_index=True, right_on="movieId")[["score", "title", "genres"]]

In [111]:
import ipywidgets as widgets
from IPython.display import display

movie_name_input = widgets.Text(
    value='Toy Story',
    description='Movie Title:',
    disabled=False
)
recommendation_list = widgets.Output()

def on_type(data):
    with recommendation_list:
        recommendation_list.clear_output()
        title = data["new"]
        if len(title) > 5:
            results = search(title)
            movie_id = results.iloc[0]["movieId"]
            display(find_similar_movies(movie_id))

movie_name_input.observe(on_type, names='value')

display(movie_name_input, recommendation_list)

Text(value='Toy Story', description='Movie Title:')

Output()

/Users/gerardbullock/Downloads/ml-25m